# Chapter 22: Degenerate Configurations

Source orientation: *Multiple View Geometry in Computer Vision*, Chapter 22, PDF pages 551-578, printed pages 533-560.

This notebook is an original computational companion to the chapter. It uses the source only for the chapter map: camera resectioning, two-view critical surfaces, Carlsson-Weinshall duality, and three-view critical configurations. It does not copy textbook prose, figures, page crops, screenshots, or exercise text.


## Chapter Goal

Degenerate configurations are not merely noisy samples. They are geometric arrangements where the images leave a whole alternative camera or reconstruction model available. By the end of the notebook you should be able to inspect a scene and ask a concrete question: which rank, nullspace, surface incidence, or homography explains why the usual estimate is not unique?

The computational translation is:

- A camera is represented by a `3 x 4` projective matrix, but many degeneracy statements depend only on its center.
- A resection ambiguity becomes a camera pencil `P_theta = P + theta P_alt`; rank drops in the pencil create linear components of the critical set.
- A two-view ambiguity becomes a ruled quadric containing the two camera centers and the scene points.
- A planar scene or pure rotation makes image correspondences live on a homography, so a fundamental matrix is no longer determined by the data in the usual way.
- A three-view ambiguity is diagnosed through the pairwise critical quadrics: usually their intersection is finite, but dependent quadrics leave a curve.


## Library Routing

| Chapter concept | Representation | Library route | Why this route | Artifact or check |
| --- | --- | --- | --- | --- |
| Planar two-view scene | Homography residuals plus eight-point singular values | `numpy`, `utils.projective`, `utils.epipolar`, Matplotlib | The degeneracy is visible both as image transfer by one homography and as extra nullity in the fundamental-matrix design matrix. | `planar-scene-homography-dominance.png`, nullity and residual checks |
| Pure rotation / critical motion | Same camera center, rotation-induced image homography | `numpy`, `scipy.spatial.transform.Rotation`, Matplotlib | With zero baseline the epipolar object loses its translation signal; the rotation homography remains observable. | `pure-rotation-critical-motion.png`, baseline and transfer checks |
| Resection critical sets | Rank and nullity of `P + theta P_alt` | `numpy.linalg`, Matplotlib | The chapter's camera-pencil proof is a rank-nullity statement, so the visual shows where the nullspace changes dimension. | `resection-pencil-rank-drop.png`, rank-drop check |
| Two-view critical surface | Ruled quadric with camera centers and points on it | Plotly 3D HTML, `numpy` | A ruled surface needs rotation and line-family inspection; Plotly exposes the generators better than a flat static panel. | `two-view-ruled-quadric-surface.html`, quadric residual check |
| Carlsson-Weinshall duality | Cremona map taking a line to a cubic trace | `numpy`, Matplotlib | The duality section is a map on projective coordinates; plotting samples and root behavior makes the exceptional tetrahedron visible. | `carlsson-weinshall-duality.png`, reference-vertex checks |
| Degeneracy diagnostics | Combined scorecard for rank, planarity, baseline, homography fit, and three-view quadric dependence | `numpy`, Matplotlib, JSON | Algorithms should report degeneracy before returning a confident model. A dashboard turns geometric failure modes into thresholds. | `degeneracy-diagnostics-dashboard.png`, `degeneracy-invariants.json` |


## Chapter-Specific Storyboard

1. **Planar scenes:** compare planar and spatial point sets under the same two cameras. The learner inspects the homography transfer and the nullspace of the eight-point matrix.
2. **Pure rotation:** keep the camera center fixed while rotating the image plane. The learner sees a valid image warp but no baseline for epipolar reconstruction.
3. **Resection pencil:** vary `theta` in `P_theta = P + theta P_alt`. The learner watches rank fall and nullity rise, matching the proof idea behind critical resection sets.
4. **Ruled quadric:** put cameras and points on a doubly ruled quadric. The learner rotates the HTML surface and checks that incidence with the quadric is the hidden reason for two-view ambiguity.
5. **Carlsson-Weinshall map:** apply the coordinate-reciprocal/Cremona map to a projective line. The learner sees why a simple line statement can dualize into a twisted-cubic statement.
6. **Diagnostics:** collect the measurable warnings into one table and one JSON file so the notebook ends with executable invariants, not only pictures.


In [ ]:
from pathlib import Path
import sys

BOOK_ROOT = Path.cwd()
for candidate in [BOOK_ROOT, *BOOK_ROOT.parents]:
    if (candidate / "00-book-index.ipynb").exists() and (candidate / "utils").exists():
        BOOK_ROOT = candidate
        break
else:
    raise RuntimeError("Could not find the MVG book root")

if str(BOOK_ROOT) not in sys.path:
    sys.path.insert(0, str(BOOK_ROOT))

TOPIC = "chapter-22"
ARTIFACT_ROOT = BOOK_ROOT / "artifacts" / TOPIC
ARTIFACT_ROOT.mkdir(parents=True, exist_ok=True)


In [ ]:
import json
import math

import matplotlib.pyplot as plt
import numpy as np
import plotly.graph_objects as go
from scipy.spatial.transform import Rotation

from utils.artifacts import assert_artifacts, display_artifact, save_json, save_matplotlib, save_plotly_html
from utils.cameras import camera_matrix, project_points, skew
from utils.projective import apply_homography, dehomogenize, dlt_homography, homogenize

plt.rcParams.update({"figure.dpi": 120, "font.size": 10})
artifact_paths = []
check_data = {}


def normalized_rank(values, *, rtol=1e-9):
    values = np.asarray(values, dtype=float)
    if values.size == 0 or values[0] == 0:
        return 0
    return int(np.sum(values > values[0] * rtol))


def eight_point_design(points1, points2):
    x1 = homogenize(points1) if np.asarray(points1).shape[-1] == 2 else np.asarray(points1, dtype=float)
    x2 = homogenize(points2) if np.asarray(points2).shape[-1] == 2 else np.asarray(points2, dtype=float)
    rows = []
    for a, b in zip(x1, x2):
        x, y, w = a
        xp, yp, wp = b
        rows.append([xp * x, xp * y, xp * w, yp * x, yp * y, yp * w, wp * x, wp * y, wp * w])
    return np.asarray(rows, dtype=float)


def transfer_error_homography(src, dst):
    H = dlt_homography(src, dst)
    mapped = apply_homography(H, src)
    return H, np.linalg.norm(mapped - dst, axis=1)


def point_grid_on_plane(nx=5, ny=4, z=4.0):
    xs = np.linspace(-1.2, 1.2, nx)
    ys = np.linspace(-0.9, 0.9, ny)
    return np.array([[x, y, z] for y in ys for x in xs], dtype=float)


def spatial_points_from_grid(nx=5, ny=4):
    pts = point_grid_on_plane(nx, ny, z=4.0)
    pts[:, 2] += 0.55 * np.sin(2.1 * pts[:, 0]) + 0.35 * np.cos(1.7 * pts[:, 1])
    return pts


def svd_profile(A):
    s = np.linalg.svd(A, compute_uv=False)
    return s / max(s[0], 1e-12), s


def nullity_from_singular_values(s, *, rtol=1e-9):
    return int(len(s) - normalized_rank(s, rtol=rtol))


def finite_camera_center(P):
    _, _, vt = np.linalg.svd(P)
    c = vt[-1]
    return c / c[-1]


def plot_image_arrows(ax, source, target, title):
    ax.scatter(source[:, 0], source[:, 1], s=18, label="view 1", color="#2b6cb0")
    ax.scatter(target[:, 0], target[:, 1], s=18, label="view 2", color="#c05621")
    for a, b in zip(source, target):
        ax.annotate("", xy=b, xytext=a, arrowprops={"arrowstyle": "->", "lw": 0.6, "color": "0.45"})
    ax.set_title(title)
    ax.set_aspect("equal", adjustable="box")
    ax.grid(True, lw=0.3, alpha=0.45)


def quadric_residual_z_equals_xy(points):
    pts = np.asarray(points, dtype=float)
    return pts[:, 2] - pts[:, 0] * pts[:, 1]


## 1. Planar Scenes: Homography Dominance

A planar scene is the most common practical degeneracy. The two images may look perfectly consistent, but they are consistent because one plane-induced homography explains all correspondences. In that case the linear equations for a fundamental matrix have extra nullity: many epipolar matrices agree with the same planar transfer.

The figure compares a plane and a non-planar point set under the same camera pair. The inspection target is the singular-value tail of the eight-point design matrix. A non-planar spatial set leaves a one-dimensional nullspace for `F`; the planar set leaves a larger nullspace because `x2 ~ H x1` already explains the measurements.


In [ ]:
P0 = np.hstack([np.eye(3), np.zeros((3, 1))])
P1 = np.hstack([np.eye(3), -np.array([[0.55], [0.08], [0.0]])])
plane_points = point_grid_on_plane()
space_points = spatial_points_from_grid()

plane_x0 = project_points(P0, plane_points)
plane_x1 = project_points(P1, plane_points)
space_x0 = project_points(P0, space_points)
space_x1 = project_points(P1, space_points)

H_plane, plane_h_errors = transfer_error_homography(plane_x0, plane_x1)
H_space, space_h_errors = transfer_error_homography(space_x0, space_x1)

A_plane = eight_point_design(plane_x0, plane_x1)
A_space = eight_point_design(space_x0, space_x1)
plane_s_rel, plane_s = svd_profile(A_plane)
space_s_rel, space_s = svd_profile(A_space)
plane_rank = normalized_rank(plane_s, rtol=1e-9)
space_rank = normalized_rank(space_s, rtol=1e-9)
plane_nullity = A_plane.shape[1] - plane_rank
space_nullity = A_space.shape[1] - space_rank

fig = plt.figure(figsize=(11.5, 7.0))
gs = fig.add_gridspec(2, 2, height_ratios=[1.0, 1.05])
ax0 = fig.add_subplot(gs[0, 0])
plot_image_arrows(ax0, plane_x0, plane_x1, "planar points: one homography transfers all samples")
ax0.legend(loc="upper right", fontsize=8)
ax1 = fig.add_subplot(gs[0, 1])
plot_image_arrows(ax1, space_x0, space_x1, "spatial points: one homography cannot fit the depth variation")
ax2 = fig.add_subplot(gs[1, 0])
indices = np.arange(1, 10)
ax2.semilogy(indices, plane_s_rel, "o-", label=f"plane nullity {plane_nullity}", color="#2b6cb0")
ax2.semilogy(indices, space_s_rel, "s-", label=f"spatial nullity {space_nullity}", color="#c05621")
ax2.set_xlabel("singular value index")
ax2.set_ylabel("relative singular value")
ax2.set_title("eight-point design matrix has extra null directions on a plane")
ax2.grid(True, which="both", lw=0.3, alpha=0.45)
ax2.legend()
ax3 = fig.add_subplot(gs[1, 1])
ax3.bar(["plane", "spatial"], [plane_h_errors.max(), space_h_errors.max()], color=["#2b6cb0", "#c05621"])
ax3.set_yscale("log")
ax3.set_ylabel("max homography transfer error")
ax3.set_title("homography fit separates plane-induced transfer from true 3D parallax")
ax3.grid(True, axis="y", which="both", lw=0.3, alpha=0.45)
fig.tight_layout()

planar_path = save_matplotlib(fig, TOPIC, "figures", "planar-scene-homography-dominance.png")
plt.close(fig)
artifact_paths.append(planar_path)
display_artifact(planar_path, width=920)

check_data["planar_scene"] = {
    "plane_homography_max_error": float(plane_h_errors.max()),
    "spatial_homography_max_error": float(space_h_errors.max()),
    "plane_eight_point_rank": int(plane_rank),
    "space_eight_point_rank": int(space_rank),
    "plane_eight_point_nullity": int(plane_nullity),
    "space_eight_point_nullity": int(space_nullity),
}
check_data["planar_scene"]


## 2. Pure Rotation As Critical Motion

Pure rotation is another way to obtain image agreement without triangulating a stable 3D structure. The camera center is unchanged, so there is no baseline and no epipolar parallax. Image points are still related by a projective warp, but the usual two-view reconstruction problem has lost the translation signal it needs.

The code uses a spatial point cloud, so the degeneracy is not caused by planarity. The only critical choice is the motion: `t = 0`. The visual compares the observed rotation warp with the singular values of the fundamental-matrix design matrix.


In [ ]:
rot = Rotation.from_euler("zyx", [18.0, -7.0, 4.0], degrees=True).as_matrix()
P_rot0 = np.hstack([np.eye(3), np.zeros((3, 1))])
P_rot1 = np.hstack([rot, np.zeros((3, 1))])
rot_points = spatial_points_from_grid(nx=6, ny=4)
rot_x0 = project_points(P_rot0, rot_points)
rot_x1 = project_points(P_rot1, rot_points)
rot_transfer = dehomogenize((rot @ homogenize(rot_x0).T).T)
rot_errors = np.linalg.norm(rot_transfer - rot_x1, axis=1)
A_rot = eight_point_design(rot_x0, rot_x1)
rot_s_rel, rot_s = svd_profile(A_rot)
rot_rank = normalized_rank(rot_s, rtol=1e-9)
rot_nullity = A_rot.shape[1] - rot_rank

P_trans = np.hstack([rot, -np.array([[0.45], [0.06], [0.02]])])
trans_x1 = project_points(P_trans, rot_points)
A_trans = eight_point_design(rot_x0, trans_x1)
trans_s_rel, trans_s = svd_profile(A_trans)
trans_rank = normalized_rank(trans_s, rtol=1e-9)
trans_nullity = A_trans.shape[1] - trans_rank
E_pure_rotation = skew(np.zeros(3)) @ rot

fig, axes = plt.subplots(1, 3, figsize=(13.2, 4.1))
plot_image_arrows(axes[0], rot_x0, rot_x1, "same center: rotation homography")
axes[0].legend(fontsize=8)
axes[1].semilogy(np.arange(1, 10), rot_s_rel, "o-", color="#805ad5", label=f"pure rotation nullity {rot_nullity}")
axes[1].semilogy(np.arange(1, 10), trans_s_rel, "s-", color="#2f855a", label=f"with baseline nullity {trans_nullity}")
axes[1].set_title("design matrix gains nullity when baseline vanishes")
axes[1].set_xlabel("singular value index")
axes[1].set_ylabel("relative singular value")
axes[1].grid(True, which="both", lw=0.3, alpha=0.45)
axes[1].legend(fontsize=8)
axes[2].axis("off")
summary_lines = [
    "motion diagnostic",
    f"baseline length: {0.0:.3g}",
    f"||[t]R||_F: {np.linalg.norm(E_pure_rotation):.3g}",
    f"rotation transfer max error: {rot_errors.max():.3g}",
    f"pure-rotation rank: {rot_rank}",
    f"translated rank: {trans_rank}",
]
axes[2].text(0.05, 0.82, "/n".join(summary_lines), va="top", family="monospace", fontsize=11)
axes[2].set_title("the image warp survives; epipolar baseline does not")
fig.tight_layout()

pure_rotation_path = save_matplotlib(fig, TOPIC, "figures", "pure-rotation-critical-motion.png")
plt.close(fig)
artifact_paths.append(pure_rotation_path)
display_artifact(pure_rotation_path, width=980)

check_data["pure_rotation"] = {
    "baseline_length": 0.0,
    "essential_frobenius_norm": float(np.linalg.norm(E_pure_rotation)),
    "rotation_homography_max_error": float(rot_errors.max()),
    "pure_rotation_eight_point_rank": int(rot_rank),
    "pure_rotation_eight_point_nullity": int(rot_nullity),
    "translated_eight_point_rank": int(trans_rank),
    "translated_eight_point_nullity": int(trans_nullity),
}
check_data["pure_rotation"]


## 3. Camera Resectioning Through A Rank-Drop Pencil

For resectioning, the chapter organizes ambiguity with the pencil `P_theta = P + theta P_alt`. If `P_theta` has rank three, its right nullspace is a single camera center. If its rank drops, the nullspace becomes a line or plane, and that linear component joins the critical set.

The example below is deliberately small: two projective cameras share the same left block but have different centers. At `theta = -1` the left block cancels. The plot is the proof idea in numerical form: rank falls, nullity rises, and the camera-center track runs toward the exceptional component.


In [ ]:
P_a = np.hstack([np.eye(3), np.zeros((3, 1))])
shift = np.array([1.0, 0.65, 0.25])
P_b = np.hstack([np.eye(3), -shift.reshape(3, 1)])
thetas = np.concatenate([np.linspace(-3.0, -1.08, 80), np.array([-1.0]), np.linspace(-0.92, 2.0, 100)])
ranks = []
nullities = []
smallest = []
center_parameter = []
center_residuals = []
for theta in thetas:
    P_theta = P_a + theta * P_b
    s = np.linalg.svd(P_theta, compute_uv=False)
    rank = normalized_rank(s, rtol=1e-10)
    ranks.append(rank)
    nullities.append(4 - rank)
    smallest.append(float(s[-1]))
    if rank == 3:
        C = finite_camera_center(P_theta)
        alpha = float(np.dot(C[:3], shift) / np.dot(shift, shift))
        center_parameter.append(np.arctan(alpha))
        center_residuals.append(float(np.linalg.norm(P_theta @ C)))
    else:
        center_parameter.append(np.nan)
        center_residuals.append(np.nan)

fig, axes = plt.subplots(1, 3, figsize=(13.6, 4.0))
axes[0].plot(thetas, ranks, color="#2b6cb0", lw=2, label="rank")
axes[0].plot(thetas, nullities, color="#c05621", lw=2, label="nullity")
axes[0].axvline(-1.0, color="0.15", lw=1, ls="--")
axes[0].set_ylim(0, 4.2)
axes[0].set_title("rank drop in the camera pencil")
axes[0].set_xlabel("theta")
axes[0].legend()
axes[0].grid(True, lw=0.3, alpha=0.45)
axes[1].semilogy(thetas, np.maximum(smallest, 1e-16), color="#805ad5", lw=2)
axes[1].axvline(-1.0, color="0.15", lw=1, ls="--")
axes[1].set_title("smallest singular value of P_theta")
axes[1].set_xlabel("theta")
axes[1].grid(True, which="both", lw=0.3, alpha=0.45)
axes[2].plot(thetas, center_parameter, color="#2f855a", lw=2)
axes[2].axvline(-1.0, color="0.15", lw=1, ls="--")
axes[2].set_title("camera center track, compressed by arctan")
axes[2].set_xlabel("theta")
axes[2].set_ylabel("arctan(position along shift)")
axes[2].grid(True, lw=0.3, alpha=0.45)
fig.tight_layout()

rank_drop_path = save_matplotlib(fig, TOPIC, "figures", "resection-pencil-rank-drop.png")
plt.close(fig)
artifact_paths.append(rank_drop_path)
display_artifact(rank_drop_path, width=980)

exception_index = int(np.where(np.isclose(thetas, -1.0))[0][0])
rank_at_exception = int(ranks[exception_index])
nullity_at_exception = int(nullities[exception_index])
finite_residuals = [r for r in center_residuals if np.isfinite(r)]
check_data["resection_pencil"] = {
    "rank_at_theta_minus_one": rank_at_exception,
    "nullity_at_theta_minus_one": nullity_at_exception,
    "rank_away_from_exception_min": int(min(r for t, r in zip(thetas, ranks) if not np.isclose(t, -1.0))),
    "max_center_nullspace_residual": float(max(finite_residuals)),
}
check_data["resection_pencil"]


## 4. Two Views: A Ruled Quadric As Critical Surface

The two-view theorem says that a critical configuration occurs when the two camera centers and all reconstructed points lie on a ruled quadric, except for special excluded placements. The purpose of the HTML artifact is not to prove the theorem by sight; it lets you inspect the hidden surface that makes the ambiguity plausible.

The chosen quadric is the doubly ruled surface `z = x y`, represented projectively by `z t - x y = 0`. Fixing `x = a` gives one family of straight generators, and fixing `y = b` gives the other. Points and camera centers are sampled on this surface, then the JSON check verifies the quadric residual.


In [ ]:
x = np.linspace(-1.35, 1.35, 55)
y = np.linspace(-1.35, 1.35, 55)
X, Y = np.meshgrid(x, y)
Z = X * Y
camera_centers_quadric = np.array([[0.95, -0.75, -0.7125], [-1.05, 0.82, -0.861]])
scene_quadric_points = np.array([
    [-1.10, -0.85, 0.935],
    [-0.75, 0.60, -0.450],
    [-0.25, -1.10, 0.275],
    [0.35, 1.05, 0.3675],
    [0.80, -0.35, -0.280],
    [1.15, 0.70, 0.805],
])
quadric_residuals = np.r_[quadric_residual_z_equals_xy(camera_centers_quadric), quadric_residual_z_equals_xy(scene_quadric_points)]

fig = go.Figure()
fig.add_trace(go.Surface(x=X, y=Y, z=Z, colorscale="Viridis", opacity=0.62, showscale=False, name="z = xy"))
for value in [-1.0, -0.35, 0.45, 1.05]:
    yy = np.linspace(-1.35, 1.35, 60)
    fig.add_trace(go.Scatter3d(x=np.full_like(yy, value), y=yy, z=value * yy, mode="lines", line={"color": "#1a365d", "width": 4}, name=f"x={value:.2f} generator"))
    xx = np.linspace(-1.35, 1.35, 60)
    fig.add_trace(go.Scatter3d(x=xx, y=np.full_like(xx, value), z=xx * value, mode="lines", line={"color": "#9c4221", "width": 4}, name=f"y={value:.2f} generator"))
fig.add_trace(go.Scatter3d(
    x=camera_centers_quadric[:, 0], y=camera_centers_quadric[:, 1], z=camera_centers_quadric[:, 2],
    mode="markers+text", marker={"size": 7, "color": "#d53f8c", "symbol": "diamond"}, text=["C1", "C2"], textposition="top center", name="camera centers"
))
fig.add_trace(go.Scatter3d(
    x=scene_quadric_points[:, 0], y=scene_quadric_points[:, 1], z=scene_quadric_points[:, 2],
    mode="markers", marker={"size": 5, "color": "#f6ad55"}, name="scene points"
))
fig.update_layout(
    title="two-view critical surface: cameras and points on a ruled quadric",
    scene={"xaxis_title": "x", "yaxis_title": "y", "zaxis_title": "z", "aspectmode": "cube"},
    margin={"l": 0, "r": 0, "t": 45, "b": 0},
    legend={"orientation": "h", "y": 0.02},
)
quadric_path = save_plotly_html(fig, TOPIC, "interactive", "two-view-ruled-quadric-surface.html")
artifact_paths.append(quadric_path)
display_artifact(quadric_path, width=900, height=620)

check_data["ruled_quadric"] = {
    "max_abs_z_minus_xy_residual": float(np.max(np.abs(quadric_residuals))),
    "camera_center_count_on_surface": int(len(camera_centers_quadric)),
    "scene_point_count_on_surface": int(len(scene_quadric_points)),
}
check_data["ruled_quadric"]


## 5. Carlsson-Weinshall Duality: A Line Becomes A Cubic Trace

The chapter's duality section uses the map

`Gamma(X, Y, Z, T) = (Y Z T, Z T X, T X Y, X Y Z)`.

Away from coordinate planes this acts like coordinate reciprocation up to scale. The useful geometric message is that a simple line statement about points and camera centers can dualize into a statement about a twisted cubic. The figure samples a projective line, marks the parameter values where one coordinate vanishes, and checks that `Gamma` sends those exceptional samples to the matching reference vertices.


In [ ]:
def line_point(lam):
    return np.array([1.0 + lam, 2.0 - lam, 3.0 + 2.0 * lam, 4.0 - lam], dtype=float)


def gamma_cw(Xh):
    Xh = np.asarray(Xh, dtype=float)
    X0, X1, X2, X3 = Xh
    return np.array([X1 * X2 * X3, X2 * X3 * X0, X3 * X0 * X1, X0 * X1 * X2], dtype=float)


lams = np.linspace(-0.75, 1.75, 180)
line_samples = np.array([line_point(l) for l in lams])
gamma_samples = np.array([gamma_cw(p) for p in line_samples])
affine_gamma = dehomogenize(gamma_samples)
roots = {
    "E1 from X=0": -1.0,
    "E2 from Y=0": 2.0,
    "E3 from Z=0": -1.5,
    "E4 from T=0": 4.0,
}
reference_vertices = np.eye(4)
root_errors = {}
for idx, (name, lam) in enumerate(roots.items()):
    mapped = gamma_cw(line_point(lam))
    mapped = mapped / np.max(np.abs(mapped)) if np.max(np.abs(mapped)) > 0 else mapped
    target = reference_vertices[idx]
    root_errors[name] = float(min(np.linalg.norm(mapped - target), np.linalg.norm(mapped + target)))

fig = plt.figure(figsize=(12.2, 4.6))
ax0 = fig.add_subplot(1, 2, 1)
coord_names = ["X", "Y", "Z", "T"]
for j, name in enumerate(coord_names):
    ax0.plot(lams, line_samples[:, j], lw=2, label=name)
for label, lam in roots.items():
    if lams.min() <= lam <= lams.max():
        ax0.axvline(lam, color="0.35", lw=0.8, ls="--")
        ax0.text(lam, ax0.get_ylim()[1] * 0.85, label.split()[0], rotation=90, va="top", ha="right", fontsize=8)
ax0.set_title("a projective line has coordinate roots")
ax0.set_xlabel("line parameter")
ax0.set_ylabel("homogeneous coordinate value")
ax0.grid(True, lw=0.3, alpha=0.45)
ax0.legend(ncol=4, fontsize=8)

ax1 = fig.add_subplot(1, 2, 2, projection="3d")
ax1.plot(affine_gamma[:, 0], affine_gamma[:, 1], affine_gamma[:, 2], color="#805ad5", lw=2)
ax1.scatter(affine_gamma[::25, 0], affine_gamma[::25, 1], affine_gamma[::25, 2], color="#d69e2e", s=18)
ax1.set_title("Gamma(line) sampled in one affine chart")
ax1.set_xlabel("x")
ax1.set_ylabel("y")
ax1.set_zlabel("z")
ax1.view_init(elev=20, azim=-58)
fig.tight_layout()

duality_path = save_matplotlib(fig, TOPIC, "figures", "carlsson-weinshall-duality.png")
plt.close(fig)
artifact_paths.append(duality_path)
display_artifact(duality_path, width=940)

check_data["carlsson_weinshall"] = {
    "reference_vertex_root_errors": root_errors,
    "sample_count": int(len(lams)),
}
check_data["carlsson_weinshall"]


## 6. Three Views And A Diagnostic Scorecard

For three views, each camera pair supplies a critical quadric. A point that participates in a genuine three-view ambiguity must survive all three pairwise tests, with an additional exclusion for the plane of the alternative camera centers. Generically three quadrics meet in finitely many points. If the three quadratic equations are linearly dependent, the intersection can remain a curve, which is the algebraic doorway to elliptic-quartic examples.

The dashboard below puts the chapter's degeneracy warnings in one place. It includes two-view rank diagnostics, pure-rotation baseline loss, a resection pencil rank drop, a ruled-quadric incidence check, and a three-view quadric-dependence rank. This is the checklist an implementation should run before trusting a reconstruction.


In [ ]:
def symmetric_quadric_vector(S):
    S = np.asarray(S, dtype=float)
    S = 0.5 * (S + S.T)
    entries = []
    for i in range(4):
        for j in range(i, 4):
            entries.append(S[i, j] if i == j else 2.0 * S[i, j])
    return np.asarray(entries, dtype=float)


Q1 = np.array([[0.0, -0.5, 0.0, 0.0], [-0.5, 0.0, 0.0, 0.0], [0.0, 0.0, 0.0, 0.5], [0.0, 0.0, 0.5, 0.0]])
Q2 = np.diag([1.0, -1.0, 0.25, 0.0])
Q3_dependent = 0.7 * Q1 - 1.3 * Q2
Q3_independent = Q3_dependent + np.diag([0.0, 0.0, 0.0, 1.0])
dependent_matrix = np.vstack([symmetric_quadric_vector(Q) for Q in [Q1, Q2, Q3_dependent]])
independent_matrix = np.vstack([symmetric_quadric_vector(Q) for Q in [Q1, Q2, Q3_independent]])
dep_s = np.linalg.svd(dependent_matrix, compute_uv=False)
ind_s = np.linalg.svd(independent_matrix, compute_uv=False)
dependent_rank = normalized_rank(dep_s, rtol=1e-10)
independent_rank = normalized_rank(ind_s, rtol=1e-10)

metrics = {
    "planar eight-point nullity": plane_nullity,
    "spatial eight-point nullity": space_nullity,
    "pure rotation baseline": 0.0,
    "translated baseline": float(np.linalg.norm([0.45, 0.06, 0.02])),
    "pencil nullity at theta=-1": nullity_at_exception,
    "ruled quadric max residual": float(np.max(np.abs(quadric_residuals))),
    "dependent quadric rank": dependent_rank,
    "independent quadric rank": independent_rank,
}

fig, axes = plt.subplots(1, 3, figsize=(14.2, 4.6))
labels = ["plane", "space", "pure rot", "baseline", "pencil", "quadric", "3Q dep", "3Q indep"]
values = [plane_nullity, space_nullity, rot_nullity, trans_nullity, nullity_at_exception, -math.log10(max(metrics["ruled quadric max residual"], 1e-16)), dependent_rank, independent_rank]
colors = ["#2b6cb0", "#c05621", "#805ad5", "#2f855a", "#d53f8c", "#718096", "#b7791f", "#4a5568"]
axes[0].bar(labels, values, color=colors)
axes[0].set_title("rank and incidence warnings")
axes[0].set_ylabel("score: nullity, rank, or -log10 residual")
axes[0].tick_params(axis="x", rotation=35)
axes[0].grid(True, axis="y", lw=0.3, alpha=0.45)
axes[1].semilogy(dep_s / dep_s[0], "o-", label=f"dependent rank {dependent_rank}", color="#b7791f")
axes[1].semilogy(ind_s / ind_s[0], "s-", label=f"independent rank {independent_rank}", color="#4a5568")
axes[1].set_title("three pairwise quadrics: coefficient rank")
axes[1].set_xlabel("singular value index")
axes[1].set_ylabel("relative singular value")
axes[1].grid(True, which="both", lw=0.3, alpha=0.45)
axes[1].legend(fontsize=8)
axes[2].axis("off")
score_text = [
    "diagnostic reading",
    f"plane H max error: {plane_h_errors.max():.2e}",
    f"spatial H max error: {space_h_errors.max():.2e}",
    f"pure rotation H max error: {rot_errors.max():.2e}",
    f"rank(P_-1): {rank_at_exception}",
    f"max |z - xy|: {np.max(np.abs(quadric_residuals)):.2e}",
    f"3-quadric dependent rank: {dependent_rank}",
]
axes[2].text(0.02, 0.92, "/n".join(score_text), va="top", family="monospace", fontsize=10.5)
axes[2].set_title("numbers to check before trusting a model")
fig.tight_layout()

diagnostics_path = save_matplotlib(fig, TOPIC, "figures", "degeneracy-diagnostics-dashboard.png")
plt.close(fig)
artifact_paths.append(diagnostics_path)
display_artifact(diagnostics_path, width=980)

check_data["three_view_quadric_diagnostics"] = {
    "dependent_quadric_coefficient_rank": int(dependent_rank),
    "independent_quadric_coefficient_rank": int(independent_rank),
    "dependent_relative_singular_values": [float(v) for v in (dep_s / dep_s[0])],
    "independent_relative_singular_values": [float(v) for v in (ind_s / ind_s[0])],
}
check_data["diagnostic_scorecard"] = {key: float(value) if isinstance(value, (np.floating, float)) else int(value) for key, value in metrics.items()}
check_json_path = save_json(check_data, TOPIC, "checks", "degeneracy-invariants.json")
artifact_paths.append(check_json_path)
display_artifact(check_json_path)
check_data["diagnostic_scorecard"]


## Applied Lab

A useful extension is to turn the diagnostics into a preflight test for your own correspondences:

1. Estimate a homography and a fundamental matrix from the same matches.
2. Compare the homography transfer error against the Sampson error of the fundamental matrix.
3. Compute the singular values of the eight-point design matrix.
4. Estimate whether the measured 3D or tracked scene points are planar by checking the smallest singular value of their centered coordinates.
5. Flag pure rotation or near-pure rotation if the estimated baseline is small relative to image noise.

The point is not to reject every planar or rotational sequence. Many useful applications are planar or rotational. The point is to route the computation to the right model and to avoid interpreting a homography-dominated sequence as a stable projective reconstruction.


In [ ]:
centered_plane = plane_points - plane_points.mean(axis=0)
centered_space = space_points - space_points.mean(axis=0)
plane_shape_s = np.linalg.svd(centered_plane, compute_uv=False)
space_shape_s = np.linalg.svd(centered_space, compute_uv=False)
plane_planarity_ratio = float(plane_shape_s[-1] / plane_shape_s[0])
space_planarity_ratio = float(space_shape_s[-1] / space_shape_s[0])

final_sanity = {
    "artifact_count": len(artifact_paths),
    "all_artifacts": [str(path.relative_to(BOOK_ROOT)) for path in artifact_paths],
    "plane_homography_max_error": float(plane_h_errors.max()),
    "spatial_homography_max_error": float(space_h_errors.max()),
    "plane_nullity": int(plane_nullity),
    "space_nullity": int(space_nullity),
    "pure_rotation_transfer_max_error": float(rot_errors.max()),
    "pure_rotation_essential_norm": float(np.linalg.norm(E_pure_rotation)),
    "pencil_rank_at_exception": int(rank_at_exception),
    "pencil_nullity_at_exception": int(nullity_at_exception),
    "ruled_quadric_max_residual": float(np.max(np.abs(quadric_residuals))),
    "cw_max_reference_vertex_error": float(max(check_data["carlsson_weinshall"]["reference_vertex_root_errors"].values())),
    "dependent_quadric_rank": int(dependent_rank),
    "independent_quadric_rank": int(independent_rank),
    "plane_planarity_ratio": plane_planarity_ratio,
    "space_planarity_ratio": space_planarity_ratio,
}

assert_artifacts(artifact_paths, min_bytes=1500)
assert final_sanity["artifact_count"] >= 7
assert final_sanity["plane_homography_max_error"] < 1e-10
assert final_sanity["spatial_homography_max_error"] > 1e-3
assert final_sanity["plane_nullity"] >= 3
assert final_sanity["space_nullity"] == 1
assert final_sanity["pure_rotation_transfer_max_error"] < 1e-10
assert final_sanity["pure_rotation_essential_norm"] < 1e-12
assert final_sanity["pencil_rank_at_exception"] == 1
assert final_sanity["pencil_nullity_at_exception"] == 3
assert final_sanity["ruled_quadric_max_residual"] < 1e-12
assert final_sanity["cw_max_reference_vertex_error"] < 1e-12
assert final_sanity["dependent_quadric_rank"] == 2
assert final_sanity["independent_quadric_rank"] == 3
assert final_sanity["plane_planarity_ratio"] < 1e-12
assert final_sanity["space_planarity_ratio"] > 1e-2
final_sanity


## Takeaways

- Degeneracy is a geometric non-uniqueness condition, not just a large residual.
- Planar scenes and pure rotations are best recognized as homography-dominated data before forcing epipolar reconstruction.
- The resectioning camera pencil turns critical sets into rank and nullity changes.
- Two-view critical configurations are organized by ruled quadrics containing the cameras and points.
- Carlsson-Weinshall duality explains why line-based statements can reappear as twisted-cubic statements after dualization.
- Three-view ambiguity can be probed by the rank and intersection behavior of the pairwise critical quadrics.
